# Read The Docs documentation available [here](https://attackbench.readthedocs.io/en/latest/index.html)

**This notebook targets AttackBenchLib 2.0.** Three things changed in a way that moves the
numbers, so results are not comparable with 1.x:

- `results['distances']` is `d*`, the smallest perturbation found *during* the attack
  (Algorithm 1 of the paper), not the sample the attack returned last — which is in
  `results['final_distances']`.
- attacks run under a **query budget** of 2000 forward+backward propagations per sample,
  the budget used in the paper; that is what makes them comparable to each other.
- `stats['accuracy']` is the clean accuracy. In 1.x it reported the fraction of samples
  the model *already* got wrong.

# Basic demo/workflow

### AttackBenchLib installation

The recommended Colab stack excludes Torchattacks because upstream 3.5.1 pins an
obsolete `requests` version. RobustBench already installs its supported AutoAttack
dependency, so no separate GitHub install is needed.

In [ ]:
!pip install -q "attackbenchlib[models,attacks]>=2.0.2,<2.1"

In [ ]:
import attackbench
attackbench.__version__

### Install adv-lib from GH if you want to use its attacks

`adv-lib` depends on `visdom`, whose `setup.py` imports `pkg_resources` — removed in
setuptools 81+ — so `visdom` has to be built against an older setuptools first.
The paper ranks AdvLib's implementations among the best, so it is worth the two extra
lines. Without it, its attacks simply do not appear in `list_attacks()`.

In [ ]:
!pip install -q "setuptools<81" wheel
!pip install -q --no-build-isolation visdom
!pip install -q "adv-lib @ git+https://github.com/jeromerony/adversarial-library"

### Optional W&B login

Set `WANDB_API_KEY` before running this cell to enable artifact downloads. The attack
workflow itself works without a W&B account.

In [ ]:
import os
import wandb

if os.environ.get('WANDB_API_KEY'):
    wandb.login()
else:
    print('W&B login skipped; artifact-backed optimality will be unavailable.')

## Import main functions for running attacks and making evaluations
(make sure torch version supports GPU on local machines)

In [ ]:
from attackbench import run_attack, get_stats, get_loader, get_model
from attackbench.attacks import apgd

In [ ]:
import torch
print(torch.__version__)

In [ ]:
# device definition
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# loading model and dataset
model = attackbench.get_model(model_name='Standard', dataset='cifar10', source='robustbench')
dataset = get_loader('cifar10', num_samples=128)

# running apgd attack — threat_model is passed to run_attack and the attack adapts;
# query_budget defaults to 2000 propagations per sample (the paper's protocol),
# pass query_budget=None to run without a limit
results = run_attack(
    model=model,
    dataset=dataset,
    attack=apgd,
    threat_model='l2',
    device=device,
)

# getting stats
stats = get_stats(results, 'l2')
print(f"Attack Success Rate: {stats['ASR']*100:.1f}%")
print(f"Clean accuracy:      {stats['accuracy']*100:.1f}%")
print(f"Mean L2 distance:    {stats['l2_mean_distance']:.4f}")

### What the run recorded

`d*` is the distance the metrics are computed on. `final_distances` holds what the attack
returned at its last iteration: the gap between the two is how much an attack throws away
by not keeping its own best result. The query counts and the failure indicators come back
with every run — they are how the paper spots incorrectly applied attacks.

In [ ]:
import numpy as np

best  = np.array(results['distances']['l2'])
last  = np.array(results['final_distances']['l2'])
solved = np.isfinite(best)

print(f"median d*            : {np.median(best[solved]):.4f}")
print(f"median last iterate  : {np.median(last[solved]):.4f}")
print(f"queries used (max)   : {max(f + b for f, b in zip(results['num_forwards'], results['num_backwards']))}"
      f" / {results['query_budget']}")
print(f"batches that raised  : {sum(results['batch_failures'])}")
print(f"samples outside [0,1]: {sum(results['box_failures'])}")

### Local Optimality

Optimality compares the attack against `a*`, the best empirical attack — the per-sample
minimum over every attack in the benchmark, published on W&B and matched to your samples
by their SHA-512 hashes. It is never derived from the attack's own distances: that would
score ~1.0 whatever the attack did, so when no reference is available the key is simply
left out with an explanation. Legacy envelopes without the 2.x protocol marker are
also rejected instead of being mixed with current `d*` results.

In [ ]:
from attackbench import compute_local_optimality

try:
    opt = compute_local_optimality(results)
except ValueError as exc:
    print(f'Optimality unavailable: {exc}')
else:
    print(f"Optimality: {opt['optimality']:.2%}")

---

# Analysis focused workflow

### Here we have an example of a workflow in which the user downloads data from the W&B database without running attacks

Only the analysis code is needed, so the attack libraries and the RobustBench model zoo
can be skipped. The core dependencies (torch, torchvision, numpy, tqdm, wandb) are still
installed — they come with the package itself. This section requires W&B artifacts
that were regenerated with AttackBench 2.x; legacy artifacts are rejected.

In [ ]:
# matplotlib is not a dependency of the library, but Colab ships it
!pip install -q "attackbenchlib[metrics]>=2.0.2,<2.1" matplotlib  # (no GPU needed)

### Download precompiled distances and optimal distances to compute optimality

In [ ]:
import numpy as np
import attackbench

dataset, threat, model, n = 'cifar10', 'l2', 'Standard', 128

def require_2x_artifact(data, name):
    if data is None:
        raise RuntimeError(
            f'{name} is unavailable or incompatible with AttackBench 2.x. '
            'Regenerate and upload it from current d* results.'
        )
    metadata = data.get('metadata', {})
    if metadata.get('protocol_version') != 2 or metadata.get('distance_semantics') != 'best_observed':
        raise RuntimeError(f'{name} lacks the AttackBench 2.x protocol markers.')
    return data

# Download precompiled distances for two attacks
apgd = require_2x_artifact(
    attackbench.download_precompiled_distances(dataset, threat, model, 'apgd', 'original', n),
    'APGD result',
)
fmn = require_2x_artifact(
    attackbench.download_precompiled_distances(dataset, threat, model, 'fmn', 'original', n),
    'FMN result',
)

# Download optimal distances (hash-based, no n_samples needed)
optimal = require_2x_artifact(
    attackbench.download_optimal_distances(dataset, threat, model),
    'Optimal-distance envelope',
)
optimal_lookup = optimal['distances'][threat]  # {sha512_hash: distance}

# Match optimal distances to each attack's samples via hashes
apgd_best = np.array([optimal_lookup[h] for h in apgd['hashes']])
fmn_best  = np.array([optimal_lookup[h] for h in fmn['hashes']])

# Compute optimality. clean_acc is the model's clean accuracy and normalises the curves.
clean_acc = np.mean(apgd['correct'])
apgd_opt = attackbench.eval_optimality(np.array(apgd['distances'][threat]), apgd_best, clean_acc)
fmn_opt  = attackbench.eval_optimality(np.array(fmn['distances'][threat]),  fmn_best, clean_acc)
print(f"APGD: {apgd_opt:.4f}  |  FMN: {fmn_opt:.4f}")

### Compute security evaluation curves

In [ ]:
import matplotlib.pyplot as plt

# Compute robust accuracy curves directly from downloaded data
apgd_curves = attackbench.compute_curves(apgd, threat)
fmn_curves  = attackbench.compute_curves(fmn, threat)

fig, ax = plt.subplots(figsize=(7, 5))

ax.step(apgd_curves['robust_accuracy_curve']['thresholds'],
        apgd_curves['robust_accuracy_curve']['robust_accuracies'],
        where='post', label=f"APGD (AUC={apgd_curves['robust_accuracy_auc']:.4f})")

ax.step(fmn_curves['robust_accuracy_curve']['thresholds'],
        fmn_curves['robust_accuracy_curve']['robust_accuracies'],
        where='post', label=f"FMN  (AUC={fmn_curves['robust_accuracy_auc']:.4f})")

ax.set_xlabel(f"Perturbation budget ε ({threat})")
ax.set_ylabel("Robust accuracy")
ax.set_title(f"Security Evaluation Curves — {model} ({dataset})")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Tip:** the artifact names follow
`dataset-threatmodel-model-attack-library-nsamples`, so you can browse what is published
directly in the W&B project. After running an attack of your own, fold it into the
published lower envelope with `attackbench.update_optimal_distances(results)` — stage 5
of the framework: adding an attack does not require re-running the previous ones.

### Distance statistics

In [ ]:
from attackbench.metrics import compute_distance_statistics

# Compute detailed distance statistics for each attack
apgd_stats = compute_distance_statistics(apgd['distances'][threat])
fmn_stats  = compute_distance_statistics(fmn['distances'][threat])

for name, stats in [('APGD', apgd_stats), ('FMN', fmn_stats)]:
    print(f"── {name} ──")
    print(f"  Mean dist:   {stats['mean_distance']:.4f}")
    print(f"  Median dist: {stats['median_distance']:.4f}")
    print(f"  Std dist:    {stats['std_distance']:.4f}")
    print(f"  Min / Max:   {stats['min_distance']:.4f} / {stats['max_distance']:.4f}")
    print(f"  P95 / P99:   {stats['p95_distance']:.4f} / {stats['p99_distance']:.4f}")
    print(f"  Success rate:{stats['success_rate']:.2%}")
    print()

Ensemble analysis

In [ ]:
from attackbench.metrics import ensemble_distances, ensemble_gain, complementarity

apgd_d = np.array(apgd['distances'][threat])
fmn_d  = np.array(fmn['distances'][threat])
apgd_s = np.array(apgd['adv_success'], dtype=bool)
fmn_s  = np.array(fmn['adv_success'], dtype=bool)

# Ensemble: element-wise best (minimum) distance
ens_d = ensemble_distances(apgd_d, fmn_d)

print(f"APGD mean dist:     {apgd_d[apgd_d < np.inf].mean():.4f}")
print(f"FMN  mean dist:     {fmn_d[fmn_d < np.inf].mean():.4f}")
print(f"Ensemble mean dist: {ens_d[ens_d < np.inf].mean():.4f}")
print()
print(f"Complementarity:    {complementarity(apgd_s, fmn_s):.4f}")
print(f"Gain APGD→FMN:      {ensemble_gain(apgd_s, fmn_s):.4f}")
print(f"Gain FMN→APGD:      {ensemble_gain(fmn_s, apgd_s):.4f}")

---

# Custom attack workflow

See docs for more detail on the custom attack signature

In [ ]:
import torch
import numpy as np
import attackbench

# --- Define a custom minimum-norm attack: random noise + binary search (L2) ---
def random_noise_attack(model, inputs, labels, n_restarts=20, search_steps=10, **kwargs):
    """Random noise with binary search to approximate minimum L2 distance."""
    batch_size = inputs.shape[0]
    best_adv = inputs.clone()
    best_dist = torch.full((batch_size,), float('inf'), device=inputs.device)

    for _ in range(n_restarts):
        eps_lo = torch.zeros(batch_size, device=inputs.device)
        eps_hi = torch.ones(batch_size, device=inputs.device) * 0.5

        for _ in range(search_steps):
            eps_mid = (eps_lo + eps_hi) / 2
            noise = torch.randn_like(inputs)
            flat = noise.flatten(1)
            noise = noise / flat.norm(dim=1, keepdim=True).view(-1, 1, 1, 1) * eps_mid.view(-1, 1, 1, 1)
            adv = torch.clamp(inputs + noise, 0.0, 1.0)

            with torch.no_grad():
                preds = model(adv).argmax(dim=1)

            success = preds != labels
            eps_hi = torch.where(success, eps_mid, eps_hi)
            eps_lo = torch.where(success, eps_lo, eps_mid)

            dist = (adv - inputs).flatten(1).norm(dim=1)
            improved = success & (dist < best_dist)
            best_adv[improved] = adv[improved]
            best_dist[improved] = dist[improved]

    return best_adv

# Wrap it with AttackBench. The query budget applies here too: every model call the
# attack makes is counted, this one spends n_restarts * search_steps = 200 forwards.
custom_attack = attackbench.create_custom_attack(
    random_noise_attack, attack_name="RandomNoise"
)

# Load model & dataset
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model   = attackbench.load_model('Standard', dataset='cifar10', threat_model='L2')
loader  = attackbench.get_loader('cifar10', num_samples=128)

# Run custom attack
results_noise = attackbench.run_attack(
    model, loader, custom_attack,
    threat_model='l2',
    device=device,
    use_cached=False,
    attack_name='random_noise',
    attack_lib='custom',
)

# Run the preconfigured APGD with its L2 default epsilon (0.5) for comparison.
from attackbench.attacks import apgd as apgd_l2
results_apgd = attackbench.run_attack(
    model, loader, apgd_l2,
    threat_model='l2',
    device=device,
)

stats_noise = attackbench.get_stats(results_noise, 'l2')
stats_apgd  = attackbench.get_stats(results_apgd, 'l2')
print(f"Random Noise — ASR: {stats_noise['ASR']*100:.1f}%  Mean L2: {stats_noise['l2_mean_distance']:.4f}")
print(f"APGD         — ASR: {stats_apgd['ASR']*100:.1f}%  Mean L2: {stats_apgd['l2_mean_distance']:.4f}")

### SEC comparison between custom attack and optimal distances

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from attackbench.metrics import compute_robust_accuracy_curve

threat = 'l2'

# Download a compatible 2.x optimal-distance envelope
optimal = attackbench.download_optimal_distances('cifar10', threat, 'Standard')
if optimal is None:
    raise RuntimeError(
        'No compatible AttackBench 2.x lower envelope is available for this setup. '
        'Regenerate and upload it from current d* results before running this cell.'
    )
optimal_lookup = optimal['distances'][threat]

# Match optimal distances to our samples via hashes
best = np.array([optimal_lookup[h] for h in results_noise['hashes']])

# Distances arrays
noise_d = np.array(results_noise['distances'][threat])
apgd_d  = np.array(results_apgd['distances'][threat])

# Optimality scores (clean accuracy comes from the run itself)
clean_acc = np.mean(results_noise['correct'])
opt_noise = attackbench.eval_optimality(noise_d, best, clean_acc)
opt_apgd  = attackbench.eval_optimality(apgd_d,  best, clean_acc)
print(f"Random Noise optimality: {opt_noise:.4f}")
print(f"APGD optimality:         {opt_apgd:.4f}")

# Security Evaluation Curves
curve_noise = compute_robust_accuracy_curve(noise_d, np.array(results_noise['adv_success']))
curve_apgd  = compute_robust_accuracy_curve(apgd_d,  np.array(results_apgd['adv_success']))
curve_best  = compute_robust_accuracy_curve(best,     np.ones(len(best), dtype=bool))

fig, ax = plt.subplots(figsize=(7, 5))
ax.step(curve_noise['thresholds'], curve_noise['robust_accuracies'],
        where='post', label=f"Random Noise (opt={opt_noise:.4f})")
ax.step(curve_apgd['thresholds'],  curve_apgd['robust_accuracies'],
        where='post', label=f"APGD (opt={opt_apgd:.4f})")
ax.step(curve_best['thresholds'],  curve_best['robust_accuracies'],
        where='post', label="Optimal (reference)", linestyle='--', color='k')

ax.set_xlabel(f"Perturbation budget ε ({threat})")
ax.set_ylabel("Robust accuracy")
ax.set_title("SEC — Random Noise vs APGD vs Optimal")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()